# Phase 3B — Conflicting-Label Investigation

**Experiment:** `phase_3b_conflicting_label_investigation`

This notebook investigates feature groups that contain both target labels in the canonical phishing dataset.

Goals:
1. Reproduce the Phase 3A dataset fingerprint and feature-group structure.
2. Isolate the conflicting feature groups.
3. Characterize group sizes and label composition.
4. Inspect representative conflicting feature vectors and source rows.
5. Quantify prevalence.
6. Compare duplicate-group-aware Random Forest evaluation with and without conflicting groups as a controlled sensitivity analysis.
7. Persist a machine-readable JSON artifact.

**Important:** No production code or canonical dataset is modified. Conflicting rows are never silently deleted; exclusion is used only for the sensitivity analysis.


In [1]:
from pathlib import Path
import hashlib, json, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "Result"

CANDIDATE_PATHS = [
    Path("data/raw/phisingData.csv"),
    Path("notebooks/data/raw/phisingData.csv"),
    Path(r"E:\Projects\Network security log triage agent\notebooks\data\raw\phisingData.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find phisingData.csv. Update CANDIDATE_PATHS.")

ARTIFACT_DIR = Path("evaluation")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_PATH = ARTIFACT_DIR / "conflicting_label_investigation.json"

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def dataframe_fingerprint(frame):
    payload = {
        "columns": list(frame.columns),
        "dtypes": [str(x) for x in frame.dtypes],
        "shape": list(frame.shape),
    }
    row_hash = pd.util.hash_pandas_object(frame, index=True).values.tobytes()
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode() + row_hash
    ).hexdigest()

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise KeyError(f"Target column {TARGET!r} not found.")

file_sha256 = sha256_file(DATA_PATH)
df_fp = dataframe_fingerprint(df)
feature_cols = [c for c in df.columns if c != TARGET]

print("Dataset:", DATA_PATH)
print("Shape:", df.shape)
print("SHA256:", file_sha256)
print("DataFrame fingerprint:", df_fp)
print("Target counts:")
print(df[TARGET].value_counts().sort_index())


Dataset: data\raw\phisingData.csv
Shape: (11055, 31)
SHA256: a4b16abbd8610e4f53fd63e8eb3da793157a961111ed5e9d9d8afa21af866995
DataFrame fingerprint: 030addabac1f69d50f23008bd055461e78747c66484da47442a6a635e18c0669
Target counts:
Result
-1    4898
 1    6157
Name: count, dtype: int64


## 1. Reconstruct feature groups

A feature group is defined by all feature columns, excluding `Result`. A group is conflicting when the same feature vector occurs with both `-1` and `1` labels.


In [2]:
grouped = (
    df.groupby(feature_cols, dropna=False, sort=False)
      .agg(
          group_size=(TARGET, "size"),
          unique_labels=(TARGET, "nunique"),
          negative_count=(TARGET, lambda s: int((s == -1).sum())),
          positive_count=(TARGET, lambda s: int((s == 1).sum())),
      )
      .reset_index()
)

conflict_table = grouped[grouped["unique_labels"] > 1].copy()
conflict_table["majority_label"] = np.where(
    conflict_table["positive_count"] >= conflict_table["negative_count"], 1, -1
)
conflict_table["minority_count"] = conflict_table[["negative_count", "positive_count"]].min(axis=1)
conflict_table["label_balance_ratio"] = conflict_table["minority_count"] / conflict_table["group_size"]

group_sizes = df.groupby(feature_cols, dropna=False, sort=False).size()
label_nunique = df.groupby(feature_cols, dropna=False, sort=False)[TARGET].nunique()

print("Feature duplicate groups:", int((group_sizes > 1).sum()))
print("Max feature group size:", int(group_sizes.max()))
print("Conflicting groups:", len(conflict_table))
print("Rows in conflicting groups:", int(conflict_table["group_size"].sum()))

if len(conflict_table) != 64:
    print("WARNING: expected 64 conflicting groups from Phase 3A; observed", len(conflict_table))


Feature duplicate groups: 2614
Max feature group size: 25
Conflicting groups: 64
Rows in conflicting groups: 357


## 2. Group-size and label-composition distributions


In [3]:
group_size_distribution = (
    conflict_table["group_size"].value_counts()
    .sort_index()
    .rename_axis("group_size")
    .reset_index(name="number_of_conflicting_groups")
)

composition_distribution = (
    conflict_table.groupby(["negative_count", "positive_count"])
    .size()
    .reset_index(name="number_of_groups")
    .sort_values(["negative_count", "positive_count"])
)

display(group_size_distribution)
display(composition_distribution)


,group_size,number_of_conflicting_groups
0,2,6
1,3,14
2,4,8
3,5,3
4,6,11
5,7,8
6,8,4
7,9,4
8,10,3
9,11,1


,negative_count,positive_count,number_of_groups
0,1,1,6
1,1,2,10
2,1,3,2
3,1,4,3
4,1,5,3
5,1,6,2
6,1,9,1
7,2,1,4
8,2,2,3
9,2,4,2


## 3. Inspect representative conflicting groups

The examples below expose the actual feature vectors and source-row indices for the largest conflicting groups. This is intended for manual investigation of possible ambiguity, encoding/collision effects, or source-label inconsistency. No automatic conclusion is made.


In [4]:
conflict_table = conflict_table.sort_values(
    ["group_size", "minority_count"], ascending=[False, False]
).reset_index(drop=True)

display(
    conflict_table[
        ["group_size", "negative_count", "positive_count",
         "majority_label", "minority_count", "label_balance_ratio"] + feature_cols
    ].head(20)
)

representative_rows = []
for rank, (_, row) in enumerate(conflict_table.head(10).iterrows(), start=1):
    mask = pd.Series(True, index=df.index)
    for c in feature_cols:
        v = row[c]
        if pd.isna(v):
            mask &= df[c].isna()
        else:
            mask &= df[c].eq(v)
    matches = df.loc[mask].copy()
    matches.insert(0, "conflict_group_rank", rank)
    matches.insert(1, "original_row_index", matches.index)
    representative_rows.append(matches)

representative_rows = pd.concat(representative_rows, ignore_index=True)
display(representative_rows)


,group_size,negative_count,positive_count,majority_label,minority_count,label_balance_ratio,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,...,RightClick,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report
0,13,2,11,1,2,0.153846,1,-1,1,1,...,1,1,1,1,1,0,-1,1,0,1
1,12,4,8,1,4,0.333333,1,-1,1,1,...,1,1,1,1,1,0,-1,1,0,1
2,11,3,8,1,3,0.272727,1,-1,1,1,...,1,1,1,-1,1,1,-1,1,0,1
3,10,5,5,1,5,0.500000,1,-1,1,1,...,1,1,1,1,1,1,-1,1,0,1
4,10,4,6,1,4,0.400000,1,-1,1,1,...,1,1,1,-1,1,0,-1,1,0,1
5,10,1,9,1,1,0.100000,1,-1,1,1,...,1,1,1,1,1,1,-1,1,0,1
6,9,4,5,1,4,0.444444,1,-1,1,1,...,1,1,1,1,-1,1,-1,1,1,1
7,9,5,4,-1,4,0.444444,1,-1,1,1,...,1,1,1,-1,1,0,-1,1,0,1
8,9,5,4,-1,4,0.444444,1,-1,1,1,...,1,1,1,-1,1,0,-1,1,0,1
9,9,2,7,1,2,0.222222,1,-1,1,1,...,1,1,1,1,1,1,-1,1,0,1


,conflict_group_rank,original_row_index,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,1,736,1,-1,1,1,1,-1,1,1,...,1,1,1,1,0,-1,1,0,1,1
1,1,791,1,-1,1,1,1,-1,1,1,...,1,1,1,1,0,-1,1,0,1,1
2,1,1470,1,-1,1,1,1,-1,1,1,...,1,1,1,1,0,-1,1,0,1,-1
3,1,1964,1,-1,1,1,1,-1,1,1,...,1,1,1,1,0,-1,1,0,1,1
4,1,2019,1,-1,1,1,1,-1,1,1,...,1,1,1,1,0,-1,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,10,2237,1,-1,1,1,1,-1,-1,1,...,1,1,1,1,1,-1,1,0,1,-1
98,10,3789,1,-1,1,1,1,-1,-1,1,...,1,1,1,1,1,-1,1,0,1,1
99,10,3963,1,-1,1,1,1,-1,-1,1,...,1,1,1,1,1,-1,1,0,1,-1
100,10,4346,1,-1,1,1,1,-1,-1,1,...,1,1,1,1,1,-1,1,0,1,1


## 4. Conflict prevalence

Prevalence is reported at group and row level separately.


In [5]:
prevalence = {
    "total_rows": int(len(df)),
    "total_feature_groups": int(len(grouped)),
    "conflicting_groups": int(len(conflict_table)),
    "conflicting_group_rate": float(len(conflict_table) / len(grouped)),
    "rows_in_conflicting_groups": int(conflict_table["group_size"].sum()),
    "conflicting_row_rate": float(conflict_table["group_size"].sum() / len(df)),
}
display(pd.DataFrame([prevalence]))


,total_rows,total_feature_groups,conflicting_groups,conflicting_group_rate,rows_in_conflicting_groups,conflicting_row_rate
0,11055,5785,64,0.011063,357,0.032293


## 5. Duplicate-group-aware evaluation

This repeats the Phase 3A evaluation configuration: `GroupShuffleSplit`, KNN imputation, and a seeded 128-tree Random Forest. Identical feature groups cannot cross the train/test boundary.


In [6]:
def make_group_ids(frame):
    tuples = frame[feature_cols].apply(tuple, axis=1)
    return pd.Series(pd.factorize(tuples, sort=False)[0], index=frame.index)

def evaluate_group_split(frame):
    X = frame[feature_cols].copy()
    y = frame[TARGET].map({-1: 0, 1: 1})
    if y.isna().any():
        raise ValueError("Unexpected target values outside {-1, 1}.")

    groups = make_group_ids(frame)
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    imputer = KNNImputer(n_neighbors=3, weights="uniform")
    X_train_t = imputer.fit_transform(X_train)
    X_test_t = imputer.transform(X_test)

    model = RandomForestClassifier(
        n_estimators=128, criterion="gini", bootstrap=True,
        max_depth=None, max_features="sqrt", random_state=RANDOM_STATE
    )
    model.fit(X_train_t, y_train)
    pred = model.predict(X_test_t)

    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    return {
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "train_target_counts": {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
        "test_target_counts": {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()},
        "shared_feature_groups": int(len(train_groups.intersection(test_groups))),
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred)),
        "recall": float(recall_score(y_test, pred)),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist(),
    }

all_group_eval = evaluate_group_split(df)
all_group_eval


{'train_rows': 8777,
 'test_rows': 2278,
 'train_target_counts': {'0': 3878, '1': 4899},
 'test_target_counts': {'0': 1020, '1': 1258},
 'shared_feature_groups': 0,
 'accuracy': 0.95171202809482,
 'f1': 0.9563838223632039,
 'precision': 0.9541139240506329,
 'recall': 0.958664546899841,
 'confusion_matrix': [[962, 58], [52, 1206]]}

## 6. Sensitivity analysis: exclude conflicting groups

This is **not** a proposed cleaning decision. It temporarily excludes rows belonging to conflicting feature groups to measure their influence on the group-aware evaluation. The canonical dataset is untouched.


In [7]:
# Keep only feature groups with exactly one observed label.
consistent_keys = grouped[grouped["unique_labels"] == 1][feature_cols]
consistent_index = df.set_index(feature_cols, drop=False).index
consistent_key_index = consistent_keys.set_index(feature_cols, drop=False).index

df_consistent_only = df.loc[consistent_index.isin(consistent_key_index)].copy()

print("Original rows:", len(df))
print("Rows after excluding conflicting groups:", len(df_consistent_only))
print("Rows excluded:", len(df) - len(df_consistent_only))

consistent_group_eval = evaluate_group_split(df_consistent_only)
consistent_group_eval


Original rows: 11055
Rows after excluding conflicting groups: 10698
Rows excluded: 357


{'train_rows': 8494,
 'test_rows': 2204,
 'train_target_counts': {'0': 3774, '1': 4720},
 'test_target_counts': {'0': 979, '1': 1225},
 'shared_feature_groups': 0,
 'accuracy': 0.9782214156079855,
 'f1': 0.9803921568627451,
 'precision': 0.9811937857726901,
 'recall': 0.9795918367346939,
 'confusion_matrix': [[956, 23], [25, 1200]]}

## 7. Compare evaluations


In [8]:
comparison = pd.DataFrame({
    "metric": ["accuracy", "f1", "precision", "recall"],
    "all_groups": [
        all_group_eval["accuracy"], all_group_eval["f1"],
        all_group_eval["precision"], all_group_eval["recall"]
    ],
    "conflicting_groups_excluded": [
        consistent_group_eval["accuracy"], consistent_group_eval["f1"],
        consistent_group_eval["precision"], consistent_group_eval["recall"]
    ],
})
comparison["difference_excluded_minus_all"] = (
    comparison["conflicting_groups_excluded"] - comparison["all_groups"]
)
display(comparison)


,metric,all_groups,conflicting_groups_excluded,difference_excluded_minus_all
0,accuracy,0.951712,0.978221,0.026509
1,f1,0.956384,0.980392,0.024008
2,precision,0.954114,0.981194,0.027080
3,recall,0.958665,0.979592,0.020927


## 8. Persist the investigation artifact

The artifact records dataset identity, conflict structure, representative groups, and the controlled evaluation comparison. It does not contain the full dataset.


In [9]:
def json_safe(v):
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    if isinstance(v, np.bool_): return bool(v)
    if pd.isna(v): return None
    return v

representative_metadata = []
for _, row in conflict_table.head(20).iterrows():
    representative_metadata.append({
        "group_size": int(row["group_size"]),
        "negative_count": int(row["negative_count"]),
        "positive_count": int(row["positive_count"]),
        "majority_label": int(row["majority_label"]),
        "minority_count": int(row["minority_count"]),
        "label_balance_ratio": float(row["label_balance_ratio"]),
        "feature_vector": {c: json_safe(row[c]) for c in feature_cols},
    })

result = {
    "name": "phase_3b_conflicting_label_investigation",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "completed",
    "production_code_modified": False,
    "canonical_dataset": {
        "path": str(DATA_PATH),
        "sha256": file_sha256,
        "shape": list(df.shape),
        "dataframe_fingerprint": df_fp,
        "target": TARGET,
        "feature_count": len(feature_cols),
        "target_counts": {str(k): int(v) for k, v in df[TARGET].value_counts().sort_index().items()},
    },
    "conflict_structure": {
        **prevalence,
        "feature_duplicate_groups": int((group_sizes > 1).sum()),
        "max_feature_group_size": int(group_sizes.max()),
        "consistent_groups": int((label_nunique == 1).sum()),
        "group_size_distribution": {
            str(k): int(v) for k, v in conflict_table["group_size"].value_counts().sort_index().items()
        },
        "label_composition_distribution": [
            {
                "negative_count": int(r["negative_count"]),
                "positive_count": int(r["positive_count"]),
                "number_of_groups": int(r["number_of_groups"]),
            }
            for _, r in composition_distribution.iterrows()
        ],
    },
    "representative_conflicting_groups": representative_metadata,
    "evaluation": {
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "preprocessing": {"imputer": "KNNImputer", "n_neighbors": 3, "weights": "uniform"},
        "model": {
            "type": "RandomForestClassifier", "n_estimators": 128,
            "criterion": "gini", "bootstrap": True, "max_depth": None,
            "max_features": "sqrt", "random_state": RANDOM_STATE
        },
        "duplicate_group_aware_all_data": all_group_eval,
        "duplicate_group_aware_conflicting_groups_excluded": consistent_group_eval,
    },
    "interpretation_checklist": [
        "Conflicting feature groups remain in the canonical dataset.",
        "Exclusion is a sensitivity analysis only.",
        "Representative conflicts require domain/source investigation before any cleaning decision.",
        "Group-aware splitting prevents identical feature groups from crossing train/test boundaries.",
        "This experiment does not establish real-world production performance.",
    ],
}

with ARTIFACT_PATH.open("w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Wrote:", ARTIFACT_PATH)
print("Bytes:", ARTIFACT_PATH.stat().st_size)


Wrote: evaluation\conflicting_label_investigation.json
Bytes: 29802


## 9. Final checkpoint

Before Phase 4:

- [ ] Dataset fingerprint matches Phase 3A.
- [ ] The expected conflicting-group count is reproduced.
- [ ] No production code was changed.
- [ ] No canonical rows were deleted or relabeled.
- [ ] Group-aware evaluation used `random_state=42`.
- [ ] The exclusion experiment is clearly separated from the canonical evaluation.
- [ ] JSON artifact was written under `evaluation/`.

**Next step:** review the conflicting examples and artifact before deciding how Phase 4 data-validation rules should treat them. Do not automatically drop or relabel conflicting groups from this experiment alone.
